# 1.4 Multimodal messages

`HumanMessage.content` is not only a string. It can be a **list of content blocks**: text, images, audio, and more, depending on the model.

This notebook:

1. Sends a text-only block (same idea as a plain string).
2. Uploads a PNG, base64-encodes it, and asks about the image.
3. Records a few seconds of audio and sends it to an audio-capable model.

Pick a model that supports the modality you send. OpenAI model list: [platform.openai.com/docs/models](https://platform.openai.com/docs/models).

## Text input

`content=[{"type": "text", "text": "..."}]` is the explicit form of a text prompt. Later cells add image/audio blocks to the same list.

Docs: [OpenAI models](https://platform.openai.com/docs/models)

In [ ]:
from dotenv import load_dotenv

load_dotenv()

In [ ]:
from langchain.agents import create_agent

agent = create_agent(
    model='gpt-5-nano',
    system_prompt="You are a science fiction writer, create a capital city at the users request.",
)

In [ ]:
from langchain.messages import HumanMessage

# Content as a list of blocks (here, one text block)
question = HumanMessage(content=[
    {"type": "text", "text": "What is the capital of The Moon?"}
])

response = agent.invoke(
    {"messages": [question]}
)

print(response['messages'][-1].content)

## Image input

1. Run the upload widget and choose a **PNG** (for example a picture of a city or landmark).
2. Run the next cells to encode bytes as base64.
3. Send **text + image** in one `HumanMessage` so the model can describe what it sees.

The widget must finish uploading before you read `uploader.value`.

In [ ]:
from ipywidgets import FileUpload
from IPython.display import display

uploader = FileUpload(accept='.png', multiple=False)
display(uploader)

In [ ]:
# Confirm a file is present before encoding
print(uploader.value)

In [ ]:
import base64

# Get the first (and only) uploaded file dict
uploaded_file = uploader.value[0]

# This is a memoryview
content_mv = uploaded_file["content"]

# Convert memoryview -> bytes
img_bytes = bytes(content_mv)  # or content_mv.tobytes()

# Now base64 encode (models expect a string, not raw bytes)
img_b64 = base64.b64encode(img_bytes).decode("utf-8")

In [ ]:
# Two blocks: instruction + image payload
multimodal_question = HumanMessage(content=[
    {"type": "text", "text": "Tell me about this capital"},
    {"type": "image", "base64": img_b64, "mime_type": "image/png"}
])

response = agent.invoke(
    {"messages": [multimodal_question]}
)

print(response['messages'][-1].content)

## Audio input

Record from the default microphone (`sounddevice`), write a WAV in memory, then base64-encode it — same pattern as the image.

Switch the agent to an **audio-capable** model (`gpt-audio` here). Text-only models will reject the audio block.

In [ ]:
import sounddevice as sd
from scipy.io.wavfile import write
import base64
import io
import time
from tqdm import tqdm

# Recording settings
duration = 5  # seconds
sample_rate = 44100

print("Recording...")
audio = sd.rec(int(duration * sample_rate), samplerate=sample_rate, channels=1)
# Progress bar for the duration
for _ in tqdm(range(duration * 10)):   # update 10× per second
    time.sleep(0.1)
sd.wait()
print("Done.")

# Write WAV to an in-memory buffer
buf = io.BytesIO()
write(buf, sample_rate, audio)
wav_bytes = buf.getvalue()

aud_b64 = base64.b64encode(wav_bytes).decode("utf-8")

In [ ]:
# Model that accepts audio content blocks
agent = create_agent(
    model='gpt-audio',
)

multimodal_question = HumanMessage(content=[
    {"type": "text", "text": "Tell me about this audio file"},
    {"type": "audio", "base64": aud_b64, "mime_type": "audio/wav"}
])

response = agent.invoke(
    {"messages": [multimodal_question]}
)

print(response['messages'][-1].content)